# Weather Type Classification - Projekt 5
## Autonomous Expert Systems and Data Exploration

### Architektura:
REST API → Raw Storage (S3) → Validation/Cleaning → Feature Engineering → Classification → Report

In [2]:
import subprocess
subprocess.run(["pip", "install", "matplotlib", "-q"])

import subprocess
subprocess.run(["pip", "install", "matplotlib", "scikit-learn", "-q"])

CompletedProcess(args=['pip', 'install', 'matplotlib', 'scikit-learn', '-q'], returncode=0)

In [3]:

import requests
import json
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

API_BASE = "https://e6uw49pbah.execute-api.us-east-1.amazonaws.com/dev"
API_TOKEN = "STUDENT_TOKEN_2026"
BUCKET = "weather-classification-projekt"
HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}

print(f"Bucket S3: {BUCKET}")

Bucket S3: weather-classification-projekt


In [4]:
# Data Ingestion from API -> Raw Storage (S3)

s3 = boto3.client('s3')

# Get available stations
response = requests.get(f"{API_BASE}/weather/stations", headers=HEADERS)
stations_data = response.json()
print("Available stations:", stations_data)

Available stations: {'stations': ['GDN_01', 'GDN_02', 'GDY_01', 'SOP_01']}


In [5]:
# Ingestion - fetch data from API and save raw JSON to S3
s3 = boto3.client('s3')
BASE_URL = "https://e6uw49pbah.execute-api.us-east-1.amazonaws.com/dev"
HEADERS = {"Authorization": "Bearer STUDENT_TOKEN_2026"}
BUCKET = "weather-classification-projekt"

stations = stations_data['stations']
all_records = []

for station_id in stations:
    url = f"{BASE_URL}/weather/batch?station_id={station_id}&limit=500"
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        data = response.json()
        records = data.get("records", data) if isinstance(data, dict) else data
        all_records.extend(records)
        print(f"{station_id}: got {len(records)} records")
    else:
        print(f"{station_id}: error {response.status_code}")

# Save raw JSON to S3
s3.put_object(
    Bucket=BUCKET,
    Key="raw/raw_weather.json",
    Body=json.dumps(all_records),
    ContentType='application/json'
)

print(f"\nTotal: {len(all_records)} records saved to s3://{BUCKET}/raw/raw_weather.json")

GDN_01: got 500 records
GDN_02: got 500 records
GDY_01: got 500 records
SOP_01: got 500 records

Total: 2000 records saved to s3://weather-classification-projekt/raw/raw_weather.json


In [27]:
# Validation, Cleaning, Rule-based Classification
obj = s3.get_object(Bucket=BUCKET, Key="raw/raw_weather.json")
raw_data = json.loads(obj['Body'].read())
df = pd.DataFrame(raw_data)

# Validation - drop rows with missing required fields
required_columns = ['temperature', 'wind_speed', 'rain_mm', 'cloud_cover', 'pressure', 'humidity']
before = len(df)
df = df.dropna(subset=required_columns)
print(f"Records after cleaning: {len(df)} (dropped {before - len(df)})")

def classify_weather(row):

    if row['pressure'] < 1005 and row['rain_mm'] > 1.0 and row['cloud_cover'] > 80:
        return 'storm-risk'
        
    elif row['rain_mm'] > 1.5:
        return 'heavy-rain'
        
    elif row['pressure'] < 1005 and row['humidity'] > 80 and row['rain_mm'] < 0.3:
        return 'unstable'
    
    elif row['rain_mm'] >= 0.3:
        return 'rainy'
    
    elif row['wind_speed'] > 9:
        return 'windy'
    
    elif row['temperature'] > 18 and row['cloud_cover'] < 50:
        return 'warm'
    
    elif row['temperature'] < 12:
        return 'cold'
    
    elif row['cloud_cover'] > 70:
        return 'cloudy'
        
    else:
        return 'stable'

df['weather_type_rule'] = df.apply(classify_weather, axis=1)
print("\nClass distribution:")
print(df['weather_type_rule'].value_counts())

# Save processed CSV to S3
s3.put_object(
    Bucket=BUCKET,
    Key="processed/labelled_weather.csv",
    Body=df.to_csv(index=False),
    ContentType='text/csv'
)
print(f"\nSaved to s3://{BUCKET}/processed/labelled_weather.csv")

Records after cleaning: 2000 (dropped 0)

Class distribution:
weather_type_rule
rainy         805
heavy-rain    277
windy         268
stable        246
cloudy        172
warm          112
unstable       79
storm-risk     41
Name: count, dtype: int64

Saved to s3://weather-classification-projekt/processed/labelled_weather.csv


In [28]:
# Model Training - Decision Tree Classifier
from io import StringIO

obj = s3.get_object(Bucket=BUCKET, Key="processed/labelled_weather.csv")
df_processed = pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')))

features = ['temperature', 'humidity', 'pressure', 'wind_speed', 'rain_mm', 'cloud_cover']
X = df_processed[features]
y = df_processed['weather_type_rule']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Training samples: 1200, Test samples: 800

Accuracy: 0.9762

Classification Report:
              precision    recall  f1-score   support

      cloudy       0.92      1.00      0.96        71
  heavy-rain       0.97      1.00      0.99       106
       rainy       0.98      1.00      0.99       309
      stable       0.98      1.00      0.99       106
  storm-risk       1.00      0.43      0.60        14
    unstable       1.00      0.69      0.81        35
        warm       0.94      1.00      0.97        45
       windy       1.00      1.00      1.00       114

    accuracy                           0.98       800
   macro avg       0.97      0.89      0.91       800
weighted avg       0.98      0.98      0.97       800



In [29]:
# Save train/test splits to S3
s3.put_object(Bucket=BUCKET, Key="splits/X_train.csv", Body=X_train.to_csv(index=False), ContentType='text/csv')
s3.put_object(Bucket=BUCKET, Key="splits/X_test.csv", Body=X_test.to_csv(index=False), ContentType='text/csv')
s3.put_object(Bucket=BUCKET, Key="splits/y_train.csv", Body=y_train.to_csv(index=False), ContentType='text/csv')
s3.put_object(Bucket=BUCKET, Key="splits/y_test.csv", Body=y_test.to_csv(index=False), ContentType='text/csv')

print("Splits saved to S3:")
print(f"  splits/X_train.csv ({len(X_train)} rows)")
print(f"  splits/X_test.csv ({len(X_test)} rows)")
print(f"  splits/y_train.csv ({len(y_train)} rows)")
print(f"  splits/y_test.csv ({len(y_test)} rows)")

Splits saved to S3:
  splits/X_train.csv (1200 rows)
  splits/X_test.csv (800 rows)
  splits/y_train.csv (1200 rows)
  splits/y_test.csv (800 rows)


In [30]:
import subprocess
subprocess.run(["pip", "install", "seaborn", "-q"])
import seaborn as sns
print("seaborn ready")

seaborn ready


In [31]:
# SECTION 5: Visualization
import matplotlib.gridspec as gridspec

CLASS_COLORS = {
    'rainy':      '#4C9BE8',
    'heavy-rain': '#2A5FA8',
    'stable':     '#6BBF8E',
    'cloudy':     '#A0A8B8',
    'warm':       '#E8A84C',
    'cold':       '#6EC6E8',
    'windy':      '#B07FD4',
    'unstable':   '#E8734C',
    'storm-risk': '#D94F4F',
}

all_classes = sorted(y.unique())
report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)

fig = plt.figure(figsize=(18, 13), facecolor='#F7F9FC')
fig.suptitle('Weather Type Classification — Visual Summary',
             fontsize=20, fontweight='bold', color='#1A2235', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig,
                       hspace=0.42, wspace=0.38,
                       left=0.06, right=0.97,
                       top=0.92, bottom=0.07)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1:])
ax3 = fig.add_subplot(gs[1, 0:2])
ax4 = fig.add_subplot(gs[1, 2])

TITLE_KW = dict(fontsize=13, fontweight='bold', color='#1A2235', pad=10)
GRID_KW  = dict(color='#E0E4EC', linewidth=0.7)

# Class distribution
counts = y.value_counts().reindex(all_classes, fill_value=0)
bars = ax1.barh(counts.index, counts.values,
                color=[CLASS_COLORS.get(c, '#888') for c in counts.index],
                edgecolor='white', linewidth=0.8)
ax1.set_title('Class Distribution (full dataset)', **TITLE_KW)
ax1.set_xlabel('Number of records', fontsize=10, color='#4A5568')
ax1.tick_params(axis='y', labelsize=10)
ax1.set_facecolor('#F7F9FC')
ax1.xaxis.grid(True, **GRID_KW)
ax1.set_axisbelow(True)
for spine in ['top', 'right', 'left']:
    ax1.spines[spine].set_visible(False)
for bar, val in zip(bars, counts.values):
    ax1.text(val + 2, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', fontsize=9, color='#4A5568')

# Precision / Recall / F1
present_classes = [c for c in all_classes if c in report and c != 'accuracy']
precision_vals = [report[c]['precision'] for c in present_classes]
recall_vals    = [report[c]['recall']    for c in present_classes]
f1_vals        = [report[c]['f1-score']  for c in present_classes]

x = np.arange(len(present_classes))
w = 0.25
ax2.bar(x - w, precision_vals, width=w, label='Precision', color='#4C9BE8', edgecolor='white')
ax2.bar(x,     recall_vals,    width=w, label='Recall',    color='#6BBF8E', edgecolor='white')
ax2.bar(x + w, f1_vals,        width=w, label='F1-score',  color='#E8A84C', edgecolor='white')
ax2.set_title('Precision / Recall / F1 per Class (test set)', **TITLE_KW)
ax2.set_ylabel('Score (0.0 – 1.0)', fontsize=10, color='#4A5568')
ax2.set_ylim(0, 1.15)
ax2.set_xticks(x)
ax2.set_xticklabels(present_classes, fontsize=10)
ax2.set_facecolor('#F7F9FC')
ax2.yaxis.grid(True, **GRID_KW)
ax2.set_axisbelow(True)
ax2.legend(fontsize=9)
for spine in ['top', 'right']:
    ax2.spines[spine].set_visible(False)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=all_classes)
cm_norm = cm.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
cm_norm = cm_norm / row_sums
sns.heatmap(cm_norm, annot=cm, fmt='d',
            xticklabels=all_classes, yticklabels=all_classes,
            cmap='Blues', linewidths=0.5, linecolor='#E0E4EC',
            cbar_kws={'shrink': 0.8},
            ax=ax3, annot_kws={'size': 11, 'weight': 'bold'})
ax3.set_title('Confusion Matrix (test set)', **TITLE_KW)
ax3.set_xlabel('Predicted label', fontsize=10, color='#4A5568')
ax3.set_ylabel('True label (rule-based)', fontsize=10, color='#4A5568')

# F1 Rules vs ML
ml_f1   = [report[c]['f1-score'] for c in present_classes]
rule_f1 = [1.0] * len(present_classes)
xr = range(len(present_classes))
wr = 0.35
ax4.bar([i - wr/2 for i in xr], rule_f1, width=wr, label='Rule-based', color='#6BBF8E', edgecolor='white')
ax4.bar([i + wr/2 for i in xr], ml_f1,   width=wr, label='Random Forest (ML)', color='#4C9BE8', edgecolor='white')
ax4.set_title('F1-score: Rules vs ML', **TITLE_KW)
ax4.set_ylabel('F1-score', fontsize=10, color='#4A5568')
ax4.set_ylim(0, 1.12)
ax4.set_xticks(list(xr))
ax4.set_xticklabels(present_classes, rotation=25, ha='right', fontsize=9)
ax4.set_facecolor('#F7F9FC')
ax4.yaxis.grid(True, **GRID_KW)
ax4.set_axisbelow(True)
ax4.legend(fontsize=9)
for spine in ['top', 'right']:
    ax4.spines[spine].set_visible(False)
for i, v in enumerate(ml_f1):
    ax4.text(i + wr/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=8, color='#1A2235')

# Save to S3
buf = __import__('io').BytesIO()
fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='#F7F9FC')
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="reports/visual_summary.png", Body=buf.getvalue(), ContentType='image/png')

plt.show()
print(f"Saved to s3://{BUCKET}/reports/visual_summary.png")

Saved to s3://weather-classification-projekt/reports/visual_summary.png


In [1]:
# Single Sample Prediction
sample = X_test.sample(1, random_state=None)
idx = sample.index[0]

prediction = model.predict(sample)[0]
true_label = y_test.loc[idx]
proba = model.predict_proba(sample)[0]
confidence = proba.max() * 100

print("=" * 40)
print("  WEATHER PREDICTION — SINGLE SAMPLE")
print("=" * 40)
print(f"  Temperature:  {sample['temperature'].values[0]:6.2f} °C")
print(f"  Humidity:     {sample['humidity'].values[0]:6.2f} %")
print(f"  Pressure:     {sample['pressure'].values[0]:6.2f} hPa")
print(f"  Wind speed:   {sample['wind_speed'].values[0]:6.2f} m/s")
print(f"  Rain:         {sample['rain_mm'].values[0]:6.2f} mm")
print(f"  Cloud cover:  {sample['cloud_cover'].values[0]:6.0f} %")
print("-" * 40)
print(f"  True label:   {true_label}")
print(f"  Prediction:   {prediction}")
print(f"  Confidence:   {confidence:.1f}%")
print(f"  Match:        {'correct' if prediction == true_label else 'wrong'}")
print("=" * 40)

NameError: name 'X_test' is not defined

In [33]:
import shutil
shutil.copy('/home/ec2-user/SageMaker/weather-classification-project.ipynb', '/tmp/notebook.ipynb')
s3.upload_file('/tmp/notebook.ipynb', BUCKET, 'notebook/weather-classification-project.ipynb')
print("Notebook saved to S3")

Notebook saved to S3
